# RBY1 Real Robot Inference (Minimal)

`inference_test.ipynb` 기준으로, **실로봇에서 필요한 최소 경로만** 남긴 버전입니다.

## 연결 구성

| 대상 | 주소 | 비고 |
|------|------|------|
| 실로봇 | `localhost:50051` | 직접 연결 |
| 시뮬레이터 | `localhost:50052` | Docker 포트 매핑 |
| 정책 서버 | `localhost:8000` | WebSocket |

### 시뮬레이터 실행 (별도 터미널)
```bash
sudo docker run --rm -it \
  -e DISPLAY=$DISPLAY \
  -v /tmp/.X11-unix:/tmp/.X11-unix \
  -v /exe \
  -p 50052:50051 \
  rainbowroboticsofficial/rby1-sim
```

## 실행 순서
1. 설정/연결 정보 로드 (`ROBOT_IP=localhost:50051`, `SIM_IP=localhost:50052`)
2. 정책 서버 메타데이터 확인
3. 초기 자세(initial pose) 이동 — 실로봇 (safe path + head)
4. RealSense 첫 프레임 확인 + env + runtime 준비
5. (cell 6) 시뮬레이터(`localhost:50052`) initial pose 이동
6. (cell 7) policy action chunk 전체를 시뮬레이터에 순차 전송
7. (cell 8/9) 안전 검사 후 실로봇에 action chunk 전송 / episode loop
8. 정리

## 주의
- 정책 서버, 시뮬레이터 Docker를 미리 실행한 뒤 진행하세요.
- 실로봇 실행 전 주변 안전을 반드시 확보하세요.


## 📦 라이브러리 Import 및 기본 설정

필요한 라이브러리를 불러오고, 로봇/Policy 서버 IP, 카메라 시리얼, 제어 파라미터 등 전역 설정값을 정의합니다.

> **항상 가장 먼저 실행해야 하는 셀입니다.**

In [ ]:
from __future__ import annotations

import logging
import time
import numpy as np
import rby1_sdk as rby

from openpi_client import action_chunk_broker
from openpi_client import websocket_client_policy as _websocket_client_policy
from openpi_client.runtime import runtime as _runtime
from openpi_client.runtime.agents import policy_agent as _policy_agent
from openpi_client.runtime import subscriber as _subscriber

from examples.rby1_real import env as _env

logging.basicConfig(level=logging.INFO, force=True)

# ---------- User Config ----------
HOST = "localhost"
PORT = 8000
ACTION_HORIZON = None  # None이면 서버 metadata(action_horizon) 사용

# 실로봇 / 시뮬레이터 연결 주소 (포트가 다르므로 겹치지 않음)
ROBOT_IP = "localhost:50051"   # 실로봇  (직접 연결)
SIM_IP   = "localhost:50052"   # 시뮬레이터 (Docker -p 50052:50052)

PROMPT = "pick up the cup and place it on the plate"

# RealSense serial
CAM_HEAD_SERIAL  = "838212070714"
CAM_LEFT_SERIAL  = "922612070040"
CAM_RIGHT_SERIAL = "838212074317"

# 테스트 모드 — True 이면 시뮬레이터/단일 action chunk 셀이 실행됨
# False 이면 해당 셀을 자동으로 건너뜁니다.
TEST_MODE = True

# Runtime/Control
MAX_HZ = 15.0
NUM_EPISODES = 1
MAX_EPISODE_STEPS = 300

ARM_COMMAND_PRIORITY = 10
ARM_MINIMUM_TIME = 10.0
LOG_ACTION_SEND = True
USE_REMOTE_GRIPPER = True

# Initial pose (safe path + head init)
ENABLE_INITIAL_POSE = True
SAFE_INIT_PATH = True
INIT_COMMAND_PRIORITY = 10
INIT_DT = 0.05
INIT_HOLD_TIME = 0.2
ENABLE_HEAD_INIT = True
HEAD_INIT_DEG = np.array([0.0, 40.0], dtype=np.float64)
HEAD_INIT = np.deg2rad(HEAD_INIT_DEG)


print("Config loaded")

print(f"ROBOT_IP={ROBOT_IP}  SIM_IP={SIM_IP}")
print(f"HOST={HOST}:{PORT}  PROMPT={PROMPT}")

## 🔗 Step 1 — Policy 서버 연결 확인

지정된 호스트(`HOST:PORT`)에서 실행 중인 Policy WebSocket 서버에 접속해 `action_horizon` 등 메타데이터를 가져옵니다.  
서버가 응답하지 않으면 이후 inference 셀이 모두 실패하므로 반드시 확인하세요.

In [ ]:
# ---------- Step 1: policy server 연결 확인 ----------
ws_client_policy = _websocket_client_policy.WebsocketClientPolicy(host=HOST, port=PORT)
server_metadata = ws_client_policy.get_server_metadata()
print("Connected to policy server metadata:", server_metadata)

model_horizon = server_metadata.get("action_horizon")
if ACTION_HORIZON is None and isinstance(model_horizon, int) and model_horizon > 0:
    action_horizon = model_horizon
else:
    action_horizon = ACTION_HORIZON if ACTION_HORIZON is not None else 50

print("Using action_horizon:", action_horizon)

## 🤖 Step 2 — 실로봇 초기 자세 이동

로봇 팔을 안전한 waypoint 경로를 거쳐 데이터 수집 시작 자세로 이동시킵니다.

- **elbow 굽힘 여부**에 따라 짧은 경로(midpoint2 → initial) 또는 전체 safe path(midpoint1 → midpoint2 → initial)를 선택합니다.
- head 초기화 및 tool flange 12V 공급도 이 단계에서 함께 수행합니다.
- 완료 시 `INITIAL_POSE_DONE = True` 로 설정됩니다.

In [ ]:
# ---------- Step 2: initial pose 이동 (safe path + head) ----------
# zero_pose.py의 elbows_bending_check 로직 참고:
#   - elbow가 굽어 있음(> 90°) → initial pose에 근접 → midpoint2 -> initial (짧은 경로)
#   - elbow가 펴져 있음(≤ 90°) → zero/straight 자세 출발 → midpoint1 -> midpoint2 -> initial (full safe path)
if ENABLE_INITIAL_POSE:
    # data-collection 초기 자세
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    # data-collection midpoint
    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    TORSO_SLICE = slice(2, 8)
    RIGHT_SLICE = slice(8, 15)
    LEFT_SLICE = slice(15, 22)
    # elbow = arm joint index 3 (0-based within arm)
    # global index: right elbow=11, left elbow=18  (head=2, torso=6, right_arm=7)
    ELBOW_INIT_THRESHOLD_DEG = 90.0  # zero_pose.py config.yaml 기본값 동일

    def _build_body_head_cmd(
        torso_q: np.ndarray | None,
        right_q: np.ndarray,
        left_q: np.ndarray,
        head_q: np.ndarray | None,
        dt: float,
    ):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )

        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )

        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )

        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(
        robot,
        stream,
        name: str,
        torso_target,
        right_target,
        left_target,
        head_target,
        duration: float,
    ):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()

        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))

        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)

        for i in range(steps):
            cmd = _build_body_head_cmd(
                torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT
            )
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)

        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(
            f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}"
        )
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))
    if ENABLE_HEAD_INIT:
        print("[init-pose] target head(deg):", np.round(HEAD_INIT_DEG, 2))

    # --- zero_pose.py elbows_bending_check 로직 ---
    # elbow = arm joint[3] (head=2, torso=6 offset 이후 → right: q[11], left: q[18])
    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now  = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(
        f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f}°, "
        f"left={np.degrees(left_elbow_now):+.1f}°  "
        f"→ {'near-INITIAL (bent)' if elbow_bent else 'near-ZERO (straight)'}"
    )

    if SAFE_INIT_PATH:
        if elbow_bent:
            # initial pose에 근접 (elbow 이미 굽음) → midpoint2만 경유해서 initial
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-initial detected: midpoint2 -> initial")
        else:
            # zero/straight 자세 출발 → 전체 safe path
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 5.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-zero detected: midpoint1 -> midpoint2 -> initial")
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]
        print("[init-pose] SAFE_INIT_PATH disabled: direct -> initial")

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None

    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init,
            stream,
            name=name,
            torso_target=torso_target,
            right_target=right_target,
            left_target=left_target,
            head_target=head_target,
            duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | final error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    # ── Tool flange 12V 공급 (그리퍼 전원) ─────────────────────────────
    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'✅ OK' if ok_v else '⚠️ 실패'}")
        time.sleep(0.5)   # 전압 안정화 대기

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

## ✋ Step 2-5 — Gripper 초기화 (Homing + 열기)

RemoteGripper UDP 클라이언트를 통해 그리퍼를 초기화합니다.

1. `initialize()` — 서버 ping 확인
2. `homing()` — 가동 범위 캘리브레이션
3. `start()` — 제어 루프 시작
4. `set_normalized_target([1.0, 1.0])` — 양쪽 그리퍼 완전 열기

> Step 2가 완료되어 tool flange 12V가 공급된 상태에서 실행하세요.  
> `USE_REMOTE_GRIPPER = False` 이면 이 셀은 건너뜁니다.

In [ ]:
# ---------- Step 2-5: Gripper 초기화 (homing + 열기) ----------
# 전제: Step 2 (initial pose) 실행 완료 → tool flange 12V 공급 완료
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 (initial pose) 셀을 실행하세요.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False → 건너뜀")
else:
    from examples.rby1_real.remote_gripper import Gripper as RemoteGripper

    # [디버그] Step 2에서 남아있는 stream 잔여 객체 제거
    # stream이 살아있으면 priority=10 control을 잡고 있어 이후 env 명령을 차단할 수 있음
    if "stream" in globals() and stream is not None:
        try:
            del stream
        except Exception:
            pass
        stream = None
        print("[gripper-init] 이전 stream 객체 정리 완료")

    # 이전 gripper 객체가 있으면 정리
    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
            print("[gripper-init] 이전 gripper 객체 stop() 완료")
        except Exception as _e:
            print(f"[gripper-init] 이전 gripper stop 무시: {_e}")
        gripper_obj = None

    print("[gripper-init] RemoteGripper 연결 중...")
    gripper_obj = RemoteGripper()
    print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

    if gripper_obj.host is None or gripper_obj.port is None:
        raise RuntimeError(
            "[gripper-init] gripper host/port가 설정되지 않았습니다.\n"
            "  examples/rby1_real/config.yaml 또는 환경변수 REMOTE_GRIPPER_HOST/PORT 확인"
        )

    # 1) ping / initialize
    print("[gripper-init] ping 확인 중...")
    ok_init = gripper_obj.initialize(verbose=True)
    print(f"[gripper-init] ping 결과: {'✅ 응답 있음' if ok_init else '❌ 응답 없음'}")
    if not ok_init:
        raise RuntimeError(
            "[gripper-init] 그리퍼 서버 응답 없음.\n"
            "  Robot PC에서 실행 확인: python gripper_server.py --port 5009"
        )

    # 2) homing (범위 캘리브레이션)
    print("[gripper-init] homing 중... (30초 이내)")
    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-init] homing 실패")

    # [디버그] homing 결과로 얻은 min_q / max_q 확인
    _min_q = getattr(gripper_obj, "min_q", None)
    _max_q = getattr(gripper_obj, "max_q", None)
    if _min_q is None or _max_q is None:
        raise RuntimeError(
            "[gripper-init] homing 성공했지만 min_q/max_q가 없음.\n"
            "  gripper_server.py 응답 형식에 min_q/max_q 필드가 있는지 확인하세요."
        )
    print(f"[gripper-init] homing 완료 | min_q={_min_q}  max_q={_max_q}")

    # 3) 제어 루프 시작
    print("[gripper-init] start() 호출 중...")
    gripper_obj.start()
    print("[gripper-init] start() 완료")

    # 4) 완전 열기 (normalized_q=1.0 → OPEN) — wait_for_reply=True로 확인
    print("[gripper-init] 완전 열기 명령 전송 (normalized=1.0)...")
    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(1.0)  # 그리퍼 물리적 이동 완료 대기 (0.3s → 1.0s)

    # [디버그] 현재 그리퍼 상태 확인
    try:
        _grip_state = gripper_obj.get_state()
        print(f"[gripper-init] 현재 그리퍼 상태: {_grip_state}  (열기 후 확인)")
    except Exception as _se:
        print(f"[gripper-init][WARN] 상태 조회 실패: {_se}")

    # [디버그] 현재 normalized target 확인
    try:
        _grip_norm = gripper_obj.get_normalized_target()
        print(f"[gripper-init] 현재 normalized target: {_grip_norm}  (1.0=OPEN 이어야 함)")
    except Exception as _ne:
        print(f"[gripper-init][WARN] normalized target 조회 실패: {_ne}")

    GRIPPER_INIT_DONE = True
    print("[gripper-init] ✅ 완료 — env 생성 시 이 객체를 재사용합니다.")


## 📷 Step 3 — RealSense 카메라 초기화 + 환경(env) 및 Runtime 구성

`MultiRealsense`를 이용해 Head / Left Wrist / Right Wrist 3대의 카메라를 동시에 초기화하고, 첫 프레임(컬러 + 뎁스)을 시각화하여 연결 상태를 확인합니다.

이후 `RBY1Environment`와 `Runtime`을 생성하여 inference 준비를 완료합니다.

| 단계 | 내용 |
|------|------|
| ① 사전 정리 | 이전 RealSense 파이프라인·env 강제 종료 |
| ② 카메라 init | MultiRealsense.start() + 첫 프레임 대기 |
| ③ 프레임 확인 | 컬러(RGB) + 뎁스(turbo colormap) 시각화 |
| ④ env/runtime | RBY1Environment + Runtime 객체 생성 |

> 완료 시 `ENV_SETUP_DONE = True` 로 설정됩니다.

In [ ]:
# ---------- Step 3: RealSense 진단 + env + runtime 준비 ----------
import sys
import gc
import importlib
import matplotlib.pyplot as plt
import pyrealsense2 as rs

# ── (A) 경량 진단: rs.context()로 연결 장치만 확인 (파이프라인 안 엶) ──────
CAM_SERIALS = {
    "head":        CAM_HEAD_SERIAL,
    "left_wrist":  CAM_LEFT_SERIAL,
    "right_wrist": CAM_RIGHT_SERIAL,
}
CAM_NAMES      = list(CAM_SERIALS.keys())
PRIMARY_SERIAL = CAM_SERIALS["head"]
IMG_WIDTH  = 640
IMG_HEIGHT = 480

realsense = None  # MultiRealsense는 더 이상 사용하지 않음

_ctx = rs.context()
_available = {
    dev.get_info(rs.camera_info.serial_number): dev.get_info(rs.camera_info.name)
    for dev in _ctx.query_devices()
}
del _ctx  # context 즉시 해제

print("─" * 60)
print(f"  [카메라 진단] 연결된 RealSense 장치: {len(_available)}개")
for serial, name in _available.items():
    cfg_name = next((n for n, s in CAM_SERIALS.items() if s == serial), "미설정")
    mark = "✅" if serial in CAM_SERIALS.values() else "ℹ️  config 없음"
    print(f"  {mark}  {cfg_name:>12}  |  {name}  |  S/N: {serial}")

_not_found = [(n, s) for n, s in CAM_SERIALS.items() if s not in _available]
if _not_found:
    print(f"\n  ❌ config에 있지만 감지 안 된 카메라:")
    for n, s in _not_found:
        print(f"      {n} ({s})  → USB 케이블/포트를 확인하세요")
    raise RuntimeError("일부 카메라가 감지되지 않습니다. USB 연결을 확인하세요.")
else:
    print(f"\n  ✅ config의 모든 카메라 ({len(CAM_SERIALS)}개)가 감지됨")
print("─" * 60)

if not _available:
    raise RuntimeError("RealSense 카메라가 하나도 감지되지 않습니다. USB 연결을 확인하세요.")

# ── (B) 기존 env / runtime 정리 ──────────────────────────────────────────
if "runtime" in globals() and runtime is not None:
    try:
        runtime.stop()
    except Exception:
        pass
    runtime = None

if "env" in globals() and env is not None:
    try:
        env.close()
        print("[cleanup] 기존 env 닫힘 ✅")
    except Exception as _e:
        print(f"[cleanup] 기존 env 닫기 실패 (무시): {_e}")
    env = None

# GC를 강제 실행해 이전 env의 _RealsenseCamera/__del__ 이 즉시 호출되도록 한다.
# 이전 실행이 Ctrl-C 등으로 중단돼 env가 미할당 상태로 남아있어도,
# env.py의 BaseException 핸들러가 카메라를 stop() 했으므로 GC로 정리된다.
gc.collect()
time.sleep(1.5)  # 장치 해제 안정화 (USB 핸드쉐이크 포함)

# ── (C) RBY1Environment 생성 (카메라는 env 내부에서 한 번만 열림) ──────────
class ActionPrintSubscriber(_subscriber.Subscriber):
    def __init__(self, every_n_steps: int = 1, print_values: bool = False) -> None:
        self._every_n = max(1, int(every_n_steps))
        self._print_values = bool(print_values)
        self._step = 0

    def on_episode_start(self) -> None:
        self._step = 0
        print("[subscriber] episode start")

    def on_step(self, observation: dict, action: dict) -> None:
        self._step += 1
        if self._step % self._every_n != 0:
            return
        actions = np.asarray(action.get("actions", []), dtype=np.float32).reshape(-1)
        if actions.size == 0:
            print(f"[action] step={self._step} | empty")
            return
        print(
            f"[action] step={self._step} | shape={actions.shape} "
            f"min={float(actions.min()):+.4f} max={float(actions.max()):+.4f} "
            f"norm={float(np.linalg.norm(actions)):.6f}"
        )
        if self._print_values:
            print("[action] values:", np.array2string(actions, precision=4, suppress_small=True))

    def on_episode_end(self) -> None:
        print("[subscriber] episode end")

# env.py 최신 변경 반영
_env = importlib.reload(_env)

env = _env.RBY1Environment(
    robot_ip=ROBOT_IP,
    prompt=PROMPT,
    render_height=224,
    render_width=224,
    camera_width=IMG_WIDTH,
    camera_height=IMG_HEIGHT,
    camera_fps=30,
    cam_head_serial=CAM_HEAD_SERIAL,
    cam_left_serial=CAM_LEFT_SERIAL,
    cam_right_serial=CAM_RIGHT_SERIAL,
    left_action_dim=8,
    right_action_dim=8,
    arm_command_priority=ARM_COMMAND_PRIORITY,
    arm_minimum_time=ARM_MINIMUM_TIME,
    log_action_send=LOG_ACTION_SEND,
    state_source="robot",
    state_zmq_address=None,
    state_indices=None,
    gripper_state_key=None,
    use_remote_gripper=USE_REMOTE_GRIPPER,
    gripper=globals().get("gripper_obj"),
)

obs = env.get_observation()
print(
    "[obs] image shapes:",
    {k: tuple(v.shape) for k, v in obs.items() if isinstance(v, np.ndarray) and "image" in k},
)
print("[obs] state shape:", np.asarray(obs["observation/state"]).shape)

# ── (D) 첫 관측 이미지 시각화 ─────────────────────────────────────────────
_img_keys = [k for k in sorted(obs.keys()) if "image" in k and isinstance(obs[k], np.ndarray)]
if _img_keys:
    _fig, _axes = plt.subplots(1, len(_img_keys), figsize=(5 * len(_img_keys), 5))
    if len(_img_keys) == 1:
        _axes = [_axes]
    for _ax, _k in zip(_axes, _img_keys):
        _img = obs[_k]
        # CHW → HWC if needed
        if _img.ndim == 3 and _img.shape[0] in (1, 3):
            _img = np.transpose(_img, (1, 2, 0))
        _ax.imshow(_img)
        _ax.set_title(_k.replace("observation/", ""), fontsize=11)
        _ax.axis("off")
    _fig.suptitle("Initial Observation (from RBY1Environment)", fontsize=13)
    plt.tight_layout()
    plt.show()

# ── (E) runtime 준비 ──────────────────────────────────────────────────────
action_subscriber = ActionPrintSubscriber(every_n_steps=1, print_values=False)

runtime = _runtime.Runtime(
    environment=env,
    agent=_policy_agent.PolicyAgent(
        policy=action_chunk_broker.ActionChunkBroker(
            policy=ws_client_policy,
            action_horizon=action_horizon,
        )
    ),
    subscribers=[action_subscriber],
    max_hz=MAX_HZ,
    num_episodes=NUM_EPISODES,
    max_episode_steps=MAX_EPISODE_STEPS,
)

ENV_SETUP_DONE = True
RUNTIME_RUNNING = False
runtime_thread = None
print("[setup] 완료: 카메라 확인 + env + runtime 준비됨")

---
## 🎮 [선택] 시뮬레이터 초기 자세 이동

`SIM_IP`에서 실행 중인 시뮬레이터 로봇을 실로봇과 동일한 초기 자세로 이동시킵니다.  
실로봇 inference 전에 시뮬레이터로 먼저 동작을 검증할 때 사용합니다.

> 시뮬레이터를 사용하지 않는 경우 건너뛰어도 됩니다.

In [ ]:
# ---------- (cell 6) 시뮬레이터 initial pose 이동 ----------
# 사용 전제: SIM_IP(설정 셀)에 시뮬레이터가 실행 중
# TEST_MODE=True 일 때만 실행, False 이면 건너뜀
if not globals().get("TEST_MODE", False):
    print("[sim-init] TEST_MODE=False — 건너뜁니다. (실행하려면 설정 셀에서 TEST_MODE=True 로 변경)")
else:
    SIM_ADDR = SIM_IP  # 설정 셀의 SIM_IP 사용
    SIM_INIT_DT = 0.05
    SIM_INIT_HOLD_TIME = 0.5

    TORSO_INIT_DEG_SIM = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG_SIM = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG_SIM  = np.array([-24.0,  60.0, -10.0, -120.0,  60.0, 85.0, 0.0], dtype=np.float64)
    MID1_RIGHT_DEG_SIM = np.array([70.0, -30.0, 0.0, -100.0,   0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG_SIM  = np.array([70.0,  30.0, 0.0, -100.0,   0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG_SIM = np.array([0.0,  -15.0, 0.0, -100.0,   0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG_SIM  = np.array([0.0,   15.0, 0.0, -100.0,   0.0, -70.0, 0.0], dtype=np.float64)

    TORSO_INIT_SIM = np.deg2rad(TORSO_INIT_DEG_SIM)
    INIT_RIGHT_SIM = np.deg2rad(INIT_RIGHT_DEG_SIM)
    INIT_LEFT_SIM  = np.deg2rad(INIT_LEFT_DEG_SIM)
    MID1_RIGHT_SIM = np.deg2rad(MID1_RIGHT_DEG_SIM)
    MID1_LEFT_SIM  = np.deg2rad(MID1_LEFT_DEG_SIM)
    MID2_RIGHT_SIM = np.deg2rad(MID2_RIGHT_DEG_SIM)
    MID2_LEFT_SIM  = np.deg2rad(MID2_LEFT_DEG_SIM)

    TORSO_SLICE_SIM = slice(2, 8)
    RIGHT_SLICE_SIM = slice(8, 15)
    LEFT_SLICE_SIM  = slice(15, 22)

    def _build_body_cmd_sim(torso_q, right_q, left_q, dt: float):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        return rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        )

    def _move_waypoint_sim(robot, stream, torso_target, right_target, left_target, duration: float):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE_SIM].copy()
        start_right = q0[RIGHT_SLICE_SIM].copy()
        start_left  = q0[LEFT_SLICE_SIM].copy()

        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / SIM_INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path  = np.linspace(start_right, right_target, num=steps)
        left_path   = np.linspace(start_left,  left_target,  num=steps)

        for i in range(steps):
            cmd_wp = _build_body_cmd_sim(torso_path[i], right_path[i], left_path[i], dt=SIM_INIT_DT)
            try:
                stream.send_command(cmd_wp)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=10)
                    stream.send_command(cmd_wp)
                else:
                    raise
            time.sleep(SIM_INIT_DT)
        return stream

    robot_sim = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim.connect()
    assert robot_sim.is_connected(), f"Failed to connect simulator at {SIM_ADDR}"

    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        raise RuntimeError("[sim-init] Failed to enable control manager")

    stream_sim = robot_sim.create_command_stream(priority=10)
    sim_waypoints = [
        (None,          MID1_RIGHT_SIM, MID1_LEFT_SIM, 5.0),
        (None,          MID2_RIGHT_SIM, MID2_LEFT_SIM, 5.0),
        (TORSO_INIT_SIM, INIT_RIGHT_SIM, INIT_LEFT_SIM, 3.0),
    ]
    for torso_t, right_t, left_t, dur_t in sim_waypoints:
        stream_sim = _move_waypoint_sim(robot_sim, stream_sim, torso_t, right_t, left_t, dur_t)

    time.sleep(0.2)
    q_init = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    print("[sim-init] q_init right:", np.round(q_init[8:15], 4))
    print("[sim-init] q_init left :", np.round(q_init[15:22], 4))

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()

    SIM_INITIAL_POSE_DONE = True
    print("[sim-init] done")

## 🧪 [선택] 시뮬레이터 단일 Action Chunk 전송 테스트

Policy inference를 1회 수행하고, 생성된 action chunk 전체를 시뮬레이터로 순차 전송합니다.  
실로봇에 전송하기 전 policy 출력이 합리적인지 시뮬레이터에서 먼저 확인하는 용도입니다.

- 전제: 시뮬레이터 초기 자세 완료 (`SIM_INITIAL_POSE_DONE = True`)
- 제어 방식: `reset_fault_control_manager()` → `enable_control_manager()` → `create_command_stream()`

In [ ]:
# ---------- (cell 7) action chunk 전체를 시뮬레이터로 순차 전송 ----------
# TEST_MODE=True 일 때만 실행, False 이면 건너뜀
if not globals().get("TEST_MODE", False):
    print("[sim-init] TEST_MODE=False — 건너뜁니다. (실행하려면 설정 셀에서 TEST_MODE=True 로 변경)")
else:
    if not globals().get("ENV_SETUP_DONE", False):
        raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
    if not globals().get("SIM_INITIAL_POSE_DONE", False):
        raise RuntimeError("먼저 cell 6(시뮬레이터 initial pose)을 실행하세요.")

    SIM_ADDR             = SIM_IP  # 설정 셀의 SIM_IP 사용
    SIM_CHUNK_DT         = 0.1
    SIM_CHUNK_HOLD_TIME  = 0.2
    SIM_CHUNK_PRIORITY   = 10

    # --------------------------------------------------
    # 1) inference
    # --------------------------------------------------
    obs_sim    = env.get_observation()
    result_sim = ws_client_policy.infer(obs_sim)
    print("[sim-chunk] inference result keys:", list(result_sim.keys()))

    actions_sim = np.asarray(result_sim["actions"], dtype=np.float64)
    if actions_sim.ndim == 1:
        assert actions_sim.size % 16 == 0, f"Expected (*,16), got {actions_sim.shape}"
        actions_sim = actions_sim.reshape(-1, 16)
    assert actions_sim.ndim == 2 and actions_sim.shape[1] == 16, f"Expected (T,16), got {actions_sim.shape}"

    right_raw = actions_sim[:, 0:7]
    left_raw  = actions_sim[:, 7:14]
    right_gripper_raw = actions_sim[:, 14]
    left_gripper_raw = actions_sim[:, 15]

    print(f"[sim-chunk] action chunk shape : {actions_sim.shape}")

    # --------------------------------------------------
    # 2) 로봇 연결 및 현재 상태 확인
    # --------------------------------------------------
    robot_sim = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim.connect()
    assert robot_sim.is_connected(), f"Failed to connect simulator at {SIM_ADDR}"

    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        raise RuntimeError("[sim-chunk] Failed to enable control manager")

    q0              = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    torso_hold      = q0[2:8].copy()    # torso: 현재 위치에서 hold
    sim_right_start = q0[8:15].copy()
    sim_left_start  = q0[15:22].copy()
    print("[sim-chunk] current right :", np.round(sim_right_start,  4))
    print("[sim-chunk] current left  :", np.round(sim_left_start,   4))

    # --------------------------------------------------
    # 3) target 생성 (policy output = absolute joint position → 그대로 사용)
    # --------------------------------------------------
    right_targets = right_raw.copy()
    left_targets  = left_raw.copy()

    print("[sim-chunk] right target[0] :", np.round(right_targets[0],  4))
    print("[sim-chunk] right target[-1]:", np.round(right_targets[-1], 4))
    print("[sim-chunk] left  target[0] :", np.round(left_targets[0],   4))
    print("[sim-chunk] left  target[-1]:", np.round(left_targets[-1],  4))

    total_cmd_norm_r = float(np.linalg.norm(right_targets[-1] - right_targets[0]))
    total_cmd_norm_l = float(np.linalg.norm(left_targets[-1]  - left_targets[0]))
    print(f"[sim-chunk] cmd total movement norm | right={total_cmd_norm_r:.4f}, left={total_cmd_norm_l:.4f}")
    if total_cmd_norm_r < 1e-3 and total_cmd_norm_l < 1e-3:
        print("[sim-chunk][WARN] 모델 output 자체 변화가 매우 작습니다. 관측이 올바른지 확인하세요.")

    # --------------------------------------------------
    # 4) stream 전송  ← stream_sim.send_command() 사용 (robot_sim 직접 호출 X)
    # --------------------------------------------------
    stream_sim  = robot_sim.create_command_stream(priority=SIM_CHUNK_PRIORITY)
    send_errors = 0

    for i in range(actions_sim.shape[0]):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(right_targets[i].astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(left_targets[i].astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
            )
        )

        try:
            stream_sim.send_command(cmd)   # ← stream_sim 통해 전송
        except RuntimeError as exc:
            send_errors += 1
            if "expired" in str(exc).lower():
                print(f"[sim-chunk][WARN] stream expired at step {i}, reconnecting...")
                robot_sim.wait_for_control_ready(1000)
                stream_sim = robot_sim.create_command_stream(priority=SIM_CHUNK_PRIORITY)
                stream_sim.send_command(cmd)
            else:
                raise

        if i == 0 or (i + 1) % 10 == 0 or (i + 1) == actions_sim.shape[0]:
            q_now   = np.asarray(robot_sim.get_state().position, dtype=np.float64)
            err_r   = float(np.linalg.norm(right_targets[i] - q_now[8:15]))
            err_l   = float(np.linalg.norm(left_targets[i]  - q_now[15:22]))
            moved_r = float(np.linalg.norm(q_now[8:15]  - sim_right_start))
            moved_l = float(np.linalg.norm(q_now[15:22] - sim_left_start))
            print(
                f"[sim-chunk] step {i+1:3d}/{actions_sim.shape[0]} "
                f"| tracking_err(r={err_r:.4f}, l={err_l:.4f}) "
                f"| moved_from_start(r={moved_r:.4f}, l={moved_l:.4f})"
            )

        time.sleep(SIM_CHUNK_DT)

    # --------------------------------------------------
    # 5) 최종 결과
    # --------------------------------------------------
    time.sleep(0.5)
    q_after = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    obs_norm_r = float(np.linalg.norm(q_after[8:15]  - sim_right_start))
    obs_norm_l = float(np.linalg.norm(q_after[15:22] - sim_left_start))
    print("\n[sim-chunk] === 결과 ===")
    print("[sim-chunk] q_after right :", np.round(q_after[8:15],  4))
    print("[sim-chunk] q_after left  :", np.round(q_after[15:22], 4))
    print(f"[sim-chunk] observed movement norm | right={obs_norm_r:.4f}, left={obs_norm_l:.4f}")
    print(f"[sim-chunk] commanded movement norm| right={total_cmd_norm_r:.4f}, left={total_cmd_norm_l:.4f}")
    print(f"[sim-chunk] send errors            : {send_errors}")

    if obs_norm_r < 1e-3 and obs_norm_l < 1e-3:
        print("[sim-chunk][WARN] 실제 이동량이 거의 없음. tracking_err 로그를 확인하세요.")

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()


## 🛡️ [선택] 안전 검사 후 실로봇 단일 Action Chunk 전송

Policy inference를 1회 수행한 뒤, 아래 3가지 안전 임계값을 검사하고 통과 시에만 실로봇으로 전송합니다.  
Episode loop 전에 단발성으로 동작을 검증하는 용도입니다.

| 검사 항목 | 임계값 변수 |
|-----------|------------|
| 전체 이동량 (L2 norm) | `SAFETY_MAX_TOTAL_NORM` |
| step 간 최대 delta | `SAFETY_MAX_STEP_DELTA` |
| 현재 위치 대비 최대 관절 변위 | `SAFETY_MAX_JOINT_DEG` |

- `SAFETY_OVERRIDE = True` 이면 임계값 초과 시 경고만 출력하고 전송 강행합니다.
- 그리퍼 명령도 함께 전송됩니다 (`USE_REMOTE_GRIPPER = True` 시).

In [ ]:
# ---------- 안전 검사 후 실로봇에 single action chunk 전송 ----------
# TEST_MODE=True 일 때만 실행, False 이면 건너뜀
if not globals().get("TEST_MODE", False):
    print("[safe-chunk] TEST_MODE=False — 건너뜁니다. (실행하려면 설정 셀에서 TEST_MODE=True 로 변경)")
else:
    if not globals().get("ENV_SETUP_DONE", False):
        raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
    if not globals().get("INITIAL_POSE_DONE", False):
        raise RuntimeError("먼저 Step 2 셀(초기 자세 이동)을 실행하세요.")

    # ======================================================
    # 안전 임계값 설정
    # ======================================================
    SAFETY_MAX_TOTAL_NORM  = 1.0    # trajectory 전체 이동량 (L2 norm, rad)
    SAFETY_MAX_STEP_DELTA  = 0.15   # 연속 스텝 간 최대 이동량 (rad)
    SAFETY_MAX_JOINT_DEG   = 30.0   # 현재 위치 대비 최대 관절 변위 (도)
    SAFETY_OVERRIDE        = True   # False=차단, True=경고 후 강제 진행

    # 전송 파라미터
    SAFE_DT        = 0.1
    SAFE_HOLD_TIME = 0.5
    SAFE_PRIORITY  = ARM_COMMAND_PRIORITY

    # ======================================================
    # 1) inference
    # ======================================================
    obs_safe    = env.get_observation()
    result_safe = ws_client_policy.infer(obs_safe)
    print("[safe-chunk] inference result keys:", list(result_safe.keys()))

    traj_safe = np.asarray(result_safe["actions"], dtype=np.float64)
    if traj_safe.ndim == 1:
        traj_safe = traj_safe.reshape(-1, 16)
    assert traj_safe.ndim == 2 and traj_safe.shape[1] == 16, f"Expected (T,16), got {traj_safe.shape}"

    right_targets_safe = traj_safe[:, 0:7]
    left_targets_safe  = traj_safe[:, 7:14]
    right_gripper_safe = traj_safe[:, 14]
    left_gripper_safe  = traj_safe[:, 15]
    T = traj_safe.shape[0]

    print(f"[safe-chunk] action shape: {traj_safe.shape}")

    # ======================================================
    # 2) 현재 로봇 상태 읽기
    # ======================================================
    robot_safe = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_safe.connect()
    assert robot_safe.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    q_safe     = np.asarray(robot_safe.get_state().position, dtype=np.float64)
    torso_hold = q_safe[2:8].copy()
    cur_right  = q_safe[8:15].copy()
    cur_left   = q_safe[15:22].copy()
    print("[safe-chunk] current right:", np.round(cur_right,  4))
    print("[safe-chunk] current left :", np.round(cur_left,   4))
    print("[safe-chunk] right target[0] :", np.round(right_targets_safe[0],  4))
    print("[safe-chunk] right target[-1]:", np.round(right_targets_safe[-1], 4))
    print("[safe-chunk] left  target[0] :", np.round(left_targets_safe[0],   4))
    print("[safe-chunk] left  target[-1]:", np.round(left_targets_safe[-1],  4))

    # ======================================================
    # 3) 안전 검사
    # ======================================================
    print("\n" + "─" * 60)
    print("[safe-chunk] ▶ 안전 검사 시작")
    violations = []

    # (a) 전체 이동량
    total_r  = float(np.linalg.norm(right_targets_safe[-1] - right_targets_safe[0]))
    total_l  = float(np.linalg.norm(left_targets_safe[-1]  - left_targets_safe[0]))
    status_a = "❌" if (total_r > SAFETY_MAX_TOTAL_NORM or total_l > SAFETY_MAX_TOTAL_NORM) else "✅"
    print(f"  {status_a} [A] 전체 이동 norm  | right={total_r:.4f}, left={total_l:.4f}  (임계={SAFETY_MAX_TOTAL_NORM:.2f} rad)")
    if total_r > SAFETY_MAX_TOTAL_NORM:
        violations.append(f"[A] right 전체 이동 norm {total_r:.4f} > {SAFETY_MAX_TOTAL_NORM:.2f} rad")
    if total_l > SAFETY_MAX_TOTAL_NORM:
        violations.append(f"[A] left  전체 이동 norm {total_l:.4f} > {SAFETY_MAX_TOTAL_NORM:.2f} rad")

    # (b) 스텝 간 최대 delta
    r_deltas = np.linalg.norm(np.diff(right_targets_safe, axis=0), axis=1)
    l_deltas = np.linalg.norm(np.diff(left_targets_safe,  axis=0), axis=1)
    max_dr   = float(r_deltas.max()) if len(r_deltas) > 0 else 0.0
    max_dl   = float(l_deltas.max()) if len(l_deltas) > 0 else 0.0
    idx_dr   = int(r_deltas.argmax()) if len(r_deltas) > 0 else 0
    idx_dl   = int(l_deltas.argmax()) if len(l_deltas) > 0 else 0
    status_b = "❌" if (max_dr > SAFETY_MAX_STEP_DELTA or max_dl > SAFETY_MAX_STEP_DELTA) else "✅"
    print(f"  {status_b} [B] 최대 스텝 delta | right={max_dr:.4f}(step {idx_dr}), left={max_dl:.4f}(step {idx_dl})  (임계={SAFETY_MAX_STEP_DELTA:.2f} rad)")
    if max_dr > SAFETY_MAX_STEP_DELTA:
        violations.append(f"[B] right step delta {max_dr:.4f} at step {idx_dr} > {SAFETY_MAX_STEP_DELTA:.2f} rad")
    if max_dl > SAFETY_MAX_STEP_DELTA:
        violations.append(f"[B] left  step delta {max_dl:.4f} at step {idx_dl} > {SAFETY_MAX_STEP_DELTA:.2f} rad")

    # (c) 현재 위치 대비 최대 관절 변위
    disp_r_deg = float(np.degrees(np.abs(right_targets_safe - cur_right[None, :]).max()))
    disp_l_deg = float(np.degrees(np.abs(left_targets_safe  - cur_left[None,  :]).max()))
    status_c   = "❌" if (disp_r_deg > SAFETY_MAX_JOINT_DEG or disp_l_deg > SAFETY_MAX_JOINT_DEG) else "✅"
    print(f"  {status_c} [C] 최대 관절 변위  | right={disp_r_deg:.2f}°, left={disp_l_deg:.2f}°  (임계={SAFETY_MAX_JOINT_DEG:.1f}°)")
    if disp_r_deg > SAFETY_MAX_JOINT_DEG:
        violations.append(f"[C] right 최대 관절 변위 {disp_r_deg:.2f}° > {SAFETY_MAX_JOINT_DEG:.1f}°")
    if disp_l_deg > SAFETY_MAX_JOINT_DEG:
        violations.append(f"[C] left  최대 관절 변위 {disp_l_deg:.2f}° > {SAFETY_MAX_JOINT_DEG:.1f}°")

    # ======================================================
    # 4) 판정
    # ======================================================
    print("─" * 60)
    if violations:
        print("🚨 안전 검사 실패 — 임계값 초과 항목:")
        for v in violations:
            print(f"   ⚠️  {v}")
        print("─" * 60)
        if not SAFETY_OVERRIDE:
            print("🔒 실행 차단됨.  SAFETY_OVERRIDE=True 로 설정 후 재실행하면 강제 전송합니다.")
            if hasattr(robot_safe, "disconnect"):
                robot_safe.disconnect()
            raise RuntimeError(f"Safety check failed ({len(violations)} violation(s)). Set SAFETY_OVERRIDE=True to override.")
        else:
            print("⚡ SAFETY_OVERRIDE=True — 경고를 무시하고 전송을 계속합니다.")
    else:
        print("✅ 안전 검사 통과 — 실로봇으로 single action chunk를 전송합니다.")
    print("─" * 60 + "\n")

    # ======================================================
    # 5) 실로봇 전송
    # ======================================================
    robot_safe.power_on(".*")
    robot_safe.servo_on(".*")
    robot_safe.reset_fault_control_manager()
    if not robot_safe.enable_control_manager():
        if hasattr(robot_safe, "disconnect"):
            robot_safe.disconnect()
        raise RuntimeError("[safe-chunk] Failed to enable control manager")

    stream_safe = robot_safe.create_command_stream(priority=SAFE_PRIORITY)
    send_errs   = 0

    for i in range(T):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(right_targets_safe[i].astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SAFE_HOLD_TIME))
                    .set_position(left_targets_safe[i].astype(np.float64))
                    .set_minimum_time(SAFE_DT)
                )
            )
        )

        try:
            stream_safe.send_command(cmd)
        except RuntimeError as exc:
            send_errs += 1
            if "expired" in str(exc).lower():
                print(f"[safe-chunk][WARN] stream expired at step {i}, reconnecting...")
                robot_safe.wait_for_control_ready(1000)
                stream_safe = robot_safe.create_command_stream(priority=SAFE_PRIORITY)
                stream_safe.send_command(cmd)
            else:
                raise

        # gripper 명령 전송
        if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
            try:
                gripper_obj.set_normalized_target(
                    np.array([float(right_gripper_safe[i]), float(left_gripper_safe[i])]),
                    wait_for_reply=False,
                )
            except Exception as _ge:
                print(f"[safe-chunk][WARN] gripper send failed: {_ge}")

        if i == 0 or (i + 1) % 10 == 0 or (i + 1) == T:
            q_now   = np.asarray(robot_safe.get_state().position, dtype=np.float64)
            err_r   = float(np.linalg.norm(right_targets_safe[i] - q_now[8:15]))
            err_l   = float(np.linalg.norm(left_targets_safe[i]  - q_now[15:22]))
            moved_r = float(np.linalg.norm(q_now[8:15]  - cur_right))
            moved_l = float(np.linalg.norm(q_now[15:22] - cur_left))
            print(
                f"[safe-chunk] step {i+1:3d}/{T} "
                f"| tracking_err(r={err_r:.4f}, l={err_l:.4f}) "
                f"| moved_from_start(r={moved_r:.4f}, l={moved_l:.4f}) "
                f"| gripper(R={right_gripper_safe[i]:+.3f}, L={left_gripper_safe[i]:+.3f})"
            )

        time.sleep(SAFE_DT)

    # ======================================================
    # 6) 최종 결과
    # ======================================================
    time.sleep(0.5)
    q_final = np.asarray(robot_safe.get_state().position, dtype=np.float64)
    obs_r   = float(np.linalg.norm(q_final[8:15]  - cur_right))
    obs_l   = float(np.linalg.norm(q_final[15:22] - cur_left))
    print(f"\n[safe-chunk] === 결과 ===")
    print(f"[safe-chunk] observed movement norm | right={obs_r:.4f}, left={obs_l:.4f}")
    print(f"[safe-chunk] commanded movement norm| right={total_r:.4f}, left={total_l:.4f}")
    print(f"[safe-chunk] send_errors            : {send_errs}")

    if obs_r < 1e-3 and obs_l < 1e-3:
        print("[safe-chunk][WARN] 실제 이동량이 거의 없음. tracking_err 로그를 확인하세요.")

    try:
        robot_safe.cancel_control()
    except Exception:
        pass
    if hasattr(robot_safe, "disconnect"):
        robot_safe.disconnect()

---
## ▶️ Step 5 — 실로봇 Inference Episode 루프 실행

Policy inference → action chunk 실행을 반복하며 전체 에피소드를 수행합니다.

- **`EPISODE_LENGTH`** : 총 실행 스텝 수
- **`EXECUTE_CHUNK_SIZE`** : inference 1회당 실제 전송 스텝 수 (나머지는 버림)
- **`CONTROL_MODE`** : `"position"` (기본) 또는 `"impedance"` 선택
  - `"position"` : `JointPositionCommandBuilder` — 기존 position control
  - `"impedance"` : `JointImpedanceControlCommandBuilder` — impedance control (compliant 동작)
- impedance 모드 파라미터: `ARM_STIFFNESS`, `ARM_DAMPING_RATIO`, `ARM_TORQUE_LIMIT`, `TORSO_STIFFNESS`, `TORSO_TORQUE_LIMIT`
- 각 스텝마다 팔 관절 + 그리퍼 명령을 동시에 전송합니다
- 루프 내에서 `_log_*` 버퍼에 commanded / actual 관절값과 카메라 이미지를 기록합니다  
  → 실행 후 아래 시각화 셀에서 분석 가능

> **실행 전 확인:** Step 2(초기 자세), Step 3(env 구성)이 완료되어 있어야 합니다.

In [ ]:
# ---------- Step 5: (실로봇) policy action replay — episode loop ----------
if not globals().get("ENV_SETUP_DONE", False):
    raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 셀(초기 자세 이동)을 실행하세요.")

# --------------------------------------------------
# 재생 파라미터
# --------------------------------------------------
REPLAY_DT          = 0.1
REPLAY_PRIORITY    = ARM_COMMAND_PRIORITY
REPLAY_HOLD_TIME   = 0.2
EPISODE_LENGTH     = 400    # 총 실행 스텝 수
EXECUTE_CHUNK_SIZE = 25     # None이면 predict chunk 전체 실행, 정수면 그 수만큼만 실행 후 재inference

# --------------------------------------------------
# 제어 모드:  "position" | "impedance"
# --------------------------------------------------
CONTROL_MODE = "position"   # ← "impedance"로 바꾸면 Joint Impedance Control 사용

# impedance 전용 파라미터 (CONTROL_MODE == "impedance" 일 때만 사용)
ARM_STIFFNESS       = np.array([80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 40.0])   # 팔 관절 강성 (Nm/rad)
ARM_DAMPING_RATIO   = 1.0                     # 감쇠비 (1.0 = critically damped)
ARM_TORQUE_LIMIT    = np.array([30.0] * 7)    # 팔 관절 토크 리밋 (Nm)
TORSO_STIFFNESS     = np.array([400.0] * 6)
TORSO_DAMPING_RATIO = 1.0
TORSO_TORQUE_LIMIT  = np.array([500.0] * 6)

assert CONTROL_MODE in ("position", "impedance"), f"Unknown CONTROL_MODE: {CONTROL_MODE}"

# --------------------------------------------------
# 로봇 연결
# --------------------------------------------------
robot = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
robot.connect()
assert robot.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

robot.power_on(".*")
robot.servo_on(".*")
robot.reset_fault_control_manager()
if not robot.enable_control_manager():
    robot.disconnect()
    raise RuntimeError("[real-replay] Failed to enable control manager")

q_init     = np.asarray(robot.get_state().position, dtype=np.float64)
ep_start_r = q_init[8:15].copy()
ep_start_l = q_init[15:22].copy()
print(f"[real-replay] EPISODE_LENGTH={EPISODE_LENGTH}, CONTROL_MODE={CONTROL_MODE}")
print(f"[real-replay] EXECUTE_CHUNK_SIZE={EXECUTE_CHUNK_SIZE} (None=full chunk)")
if CONTROL_MODE == "impedance":
    print(f"[real-replay] impedance params | arm_stiffness={ARM_STIFFNESS[0]}, damping={ARM_DAMPING_RATIO}, torque_limit={ARM_TORQUE_LIMIT[0]}")
print(f"[real-replay] episode start | right={np.round(ep_start_r, 4)}, left={np.round(ep_start_l, 4)}")

stream      = robot.create_command_stream(priority=REPLAY_PRIORITY)
total_steps = 0
chunk_idx   = 0
send_errors = 0

# --------------------------------------------------
# 로깅 버퍼
# --------------------------------------------------
_log_cmd_right    = []
_log_cmd_left     = []
_log_cmd_grip_r   = []
_log_cmd_grip_l   = []
_log_actual_right = []
_log_actual_left  = []
_log_actual_step  = []
_log_chunk_start  = []
_IMG_KEYS = ["observation/head_image", "observation/left_wrist_image", "observation/right_wrist_image"]
_log_obs_images = []

# --------------------------------------------------
# 커맨드 빌더 헬퍼
# --------------------------------------------------
def _build_arm_command(position, is_impedance):
    """팔 관절 명령 빌더 — position / impedance 모드 분기"""
    if is_impedance:
        return (
            rby.JointImpedanceControlCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
            .set_stiffness(ARM_STIFFNESS)
            .set_damping_ratio(ARM_DAMPING_RATIO)
            .set_torque_limit(ARM_TORQUE_LIMIT)
        )
    else:
        return (
            rby.JointPositionCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
        )

def _build_torso_command(position, is_impedance):
    """토르소 관절 명령 빌더 — position / impedance 모드 분기"""
    if is_impedance:
        return (
            rby.JointImpedanceControlCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
            .set_stiffness(TORSO_STIFFNESS)
            .set_damping_ratio(TORSO_DAMPING_RATIO)
            .set_torque_limit(TORSO_TORQUE_LIMIT)
        )
    else:
        return (
            rby.JointPositionCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
        )

_use_impedance = (CONTROL_MODE == "impedance")

# --------------------------------------------------
# Episode loop: infer → send chunk → repeat
# --------------------------------------------------
while total_steps < EPISODE_LENGTH:
    remaining = EPISODE_LENGTH - total_steps

    # 1) observation 취득 + inference
    obs_ep = env.get_observation()

    # policy 입력 RGB 이미지 저장
    _img_snap = {"step": total_steps}
    for _k in _IMG_KEYS:
        if _k in obs_ep:
            _img_snap[_k] = np.transpose(obs_ep[_k], (1, 2, 0))  # (C,H,W) → (H,W,C)
    _log_obs_images.append(_img_snap)

    result_ep = ws_client_policy.infer(obs_ep)

    traj16 = np.asarray(result_ep["actions"], dtype=np.float64)
    if traj16.ndim == 1:
        assert traj16.size % 16 == 0, f"Expected (*,16), got {traj16.shape}"
        traj16 = traj16.reshape(-1, 16)
    assert traj16.ndim == 2 and traj16.shape[1] == 16, f"Expected (T,16), got {traj16.shape}"

    # absolute joint position → 그대로 target
    right_targets     = traj16[:, 0:7]
    left_targets      = traj16[:, 7:14]
    right_gripper_out = traj16[:, 14]   # policy output: 0.0=CLOSE, 1.0=OPEN (서버에서 반전 처리)
    left_gripper_out  = traj16[:, 15]  # policy output: 0.0=CLOSE, 1.0=OPEN (서버에서 반전 처리)

    # 2) torso hold
    q_now      = np.asarray(robot.get_state().position, dtype=np.float64)
    torso_hold = q_now[2:8].copy()

    chunk_size    = traj16.shape[0]
    execute_size  = EXECUTE_CHUNK_SIZE if EXECUTE_CHUNK_SIZE is not None else chunk_size
    steps_to_send = min(execute_size, remaining)
    total_cmd_norm = float(np.linalg.norm(right_targets[steps_to_send-1] - right_targets[0]))

    _log_chunk_start.append(total_steps)
    print(
        f"[real-replay] chunk {chunk_idx+1} | "
        f"steps {total_steps+1}~{total_steps+steps_to_send}/{EPISODE_LENGTH} "
        f"(execute {steps_to_send}/{chunk_size} predicted) | cmd_norm={total_cmd_norm:.4f}"
    )

    # 3) chunk 전송
    for i in range(steps_to_send):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(_build_torso_command(torso_hold, _use_impedance))
                .set_right_arm_command(_build_arm_command(right_targets[i], _use_impedance))
                .set_left_arm_command(_build_arm_command(left_targets[i], _use_impedance))
            )
        )

        try:
            stream.send_command(cmd)
        except RuntimeError as exc:
            send_errors += 1
            if "expired" in str(exc).lower():
                print(f"[real-replay][WARN] stream expired at step {total_steps}, reconnecting...")
                robot.wait_for_control_ready(1000)
                stream = robot.create_command_stream(priority=REPLAY_PRIORITY)
                stream.send_command(cmd)
            else:
                raise

        # gripper 명령 전송 — policy output을 그대로 사용
        # gripper.py GRIPPER_DIRECTION=False 이므로: 0.0→CLOSE, 1.0→OPEN
        if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
            try:
                gripper_obj.set_normalized_target(
                    np.array([float(right_gripper_out[i]), float(left_gripper_out[i])]),
                    wait_for_reply=False,
                )
            except Exception as _ge:
                print(f"[real-replay][WARN] gripper send failed: {_ge}")

        # 명령 로깅
        _log_cmd_right.append(right_targets[i].copy())
        _log_cmd_left.append(left_targets[i].copy())
        _log_cmd_grip_r.append(float(right_gripper_out[i]))
        _log_cmd_grip_l.append(float(left_gripper_out[i]))

        # 중간 진단
        if i == 0 or (i + 1) % 10 == 0 or (i + 1) == steps_to_send:
            q_diag  = np.asarray(robot.get_state().position, dtype=np.float64)
            err_r   = float(np.linalg.norm(right_targets[i] - q_diag[8:15]))
            err_l   = float(np.linalg.norm(left_targets[i]  - q_diag[15:22]))
            moved_r = float(np.linalg.norm(q_diag[8:15]  - ep_start_r))
            moved_l = float(np.linalg.norm(q_diag[15:22] - ep_start_l))
            _log_actual_right.append(q_diag[8:15].copy())
            _log_actual_left.append(q_diag[15:22].copy())
            _log_actual_step.append(total_steps + i)
            mark = " [IMP]" if _use_impedance else ""
            print(
                f"  step {total_steps+i+1:4d}/{EPISODE_LENGTH}{mark} "
                f"| tracking_err(r={err_r:.4f}, l={err_l:.4f}) "
                f"| moved_from_ep_start(r={moved_r:.4f}, l={moved_l:.4f}) "
                f"| gripper(R={right_gripper_out[i]:+.3f}, L={left_gripper_out[i]:+.3f})"
            )

        time.sleep(REPLAY_DT)

    total_steps += steps_to_send
    chunk_idx   += 1

# --------------------------------------------------
# 최종 결과
# --------------------------------------------------
time.sleep(0.5)
q_after    = np.asarray(robot.get_state().position, dtype=np.float64)
obs_norm_r = float(np.linalg.norm(q_after[8:15]  - ep_start_r))
obs_norm_l = float(np.linalg.norm(q_after[15:22] - ep_start_l))

print(f"\n[real-replay] === 에피소드 완료 ({CONTROL_MODE} mode) ===")
print(f"[real-replay] total_steps={total_steps}, chunks={chunk_idx}, send_errors={send_errors}")
print(f"[real-replay] ep_start right : {np.round(ep_start_r, 4)}")
print(f"[real-replay] q_after  right : {np.round(q_after[8:15],  4)}")
print(f"[real-replay] ep_start left  : {np.round(ep_start_l, 4)}")
print(f"[real-replay] q_after  left  : {np.round(q_after[15:22], 4)}")
print(f"[real-replay] total movement norm | right={obs_norm_r:.4f}, left={obs_norm_l:.4f}")
print(f"[real-replay] 로그: {len(_log_cmd_right)} steps, {len(_log_chunk_start)} chunks → 다음 셀에서 plot 가능")

try:
    robot.cancel_control()
except Exception:
    pass

if hasattr(robot, "disconnect"):
    robot.disconnect()

# [Inital Pose]

In [ ]:
# ---------- Step 2: initial pose 이동 (safe path + head) ----------
# zero_pose.py의 elbows_bending_check 로직 참고:
#   - elbow가 굽어 있음(> 90°) → initial pose에 근접 → midpoint2 -> initial (짧은 경로)
#   - elbow가 펴져 있음(≤ 90°) → zero/straight 자세 출발 → midpoint1 -> midpoint2 -> initial (full safe path)
if ENABLE_INITIAL_POSE:
    # data-collection 초기 자세
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    # data-collection midpoint
    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    TORSO_SLICE = slice(2, 8)
    RIGHT_SLICE = slice(8, 15)
    LEFT_SLICE = slice(15, 22)
    # elbow = arm joint index 3 (0-based within arm)
    # global index: right elbow=11, left elbow=18  (head=2, torso=6, right_arm=7)
    ELBOW_INIT_THRESHOLD_DEG = 90.0  # zero_pose.py config.yaml 기본값 동일

    def _build_body_head_cmd(
        torso_q: np.ndarray | None,
        right_q: np.ndarray,
        left_q: np.ndarray,
        head_q: np.ndarray | None,
        dt: float,
    ):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )

        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )

        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )

        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(
        robot,
        stream,
        name: str,
        torso_target,
        right_target,
        left_target,
        head_target,
        duration: float,
    ):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()

        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))

        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)

        for i in range(steps):
            cmd = _build_body_head_cmd(
                torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT
            )
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)

        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(
            f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}"
        )
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))
    if ENABLE_HEAD_INIT:
        print("[init-pose] target head(deg):", np.round(HEAD_INIT_DEG, 2))

    # --- zero_pose.py elbows_bending_check 로직 ---
    # elbow = arm joint[3] (head=2, torso=6 offset 이후 → right: q[11], left: q[18])
    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now  = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(
        f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f}°, "
        f"left={np.degrees(left_elbow_now):+.1f}°  "
        f"→ {'near-INITIAL (bent)' if elbow_bent else 'near-ZERO (straight)'}"
    )

    if SAFE_INIT_PATH:
        if elbow_bent:
            # initial pose에 근접 (elbow 이미 굽음) → midpoint2만 경유해서 initial
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-initial detected: midpoint2 -> initial")
        else:
            # zero/straight 자세 출발 → 전체 safe path
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 7.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 7.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 2.0),
            ]
            print("[init-pose] near-zero detected: midpoint1 -> midpoint2 -> initial")
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]
        print("[init-pose] SAFE_INIT_PATH disabled: direct -> initial")

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None

    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init,
            stream,
            name=name,
            torso_target=torso_target,
            right_target=right_target,
            left_target=left_target,
            head_target=head_target,
            duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | final error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    # ── Tool flange 12V 공급 (그리퍼 전원) ─────────────────────────────
    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'✅ OK' if ok_v else '⚠️ 실패'}")
        time.sleep(0.5)   # 전압 안정화 대기

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

# [Gripper Initialization]

In [ ]:
# ---------- Step 2-5: Gripper 초기화 (homing + 열기) ----------
# 전제: Step 2 (initial pose) 실행 완료 → tool flange 12V 공급 완료
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 (initial pose) 셀을 실행하세요.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False → 건너뜀")
else:
    from examples.rby1_real.remote_gripper import Gripper as RemoteGripper

    # [디버그] Step 2에서 남아있는 stream 잔여 객체 제거
    # stream이 살아있으면 priority=10 control을 잡고 있어 이후 env 명령을 차단할 수 있음
    if "stream" in globals() and stream is not None:
        try:
            del stream
        except Exception:
            pass
        stream = None
        print("[gripper-init] 이전 stream 객체 정리 완료")

    # 이전 gripper 객체가 있으면 정리
    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
            print("[gripper-init] 이전 gripper 객체 stop() 완료")
        except Exception as _e:
            print(f"[gripper-init] 이전 gripper stop 무시: {_e}")
        gripper_obj = None

    print("[gripper-init] RemoteGripper 연결 중...")
    gripper_obj = RemoteGripper()
    print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

    if gripper_obj.host is None or gripper_obj.port is None:
        raise RuntimeError(
            "[gripper-init] gripper host/port가 설정되지 않았습니다.\n"
            "  examples/rby1_real/config.yaml 또는 환경변수 REMOTE_GRIPPER_HOST/PORT 확인"
        )

    # 1) ping / initialize
    print("[gripper-init] ping 확인 중...")
    ok_init = gripper_obj.initialize(verbose=True)
    print(f"[gripper-init] ping 결과: {'✅ 응답 있음' if ok_init else '❌ 응답 없음'}")
    if not ok_init:
        raise RuntimeError(
            "[gripper-init] 그리퍼 서버 응답 없음.\n"
            "  Robot PC에서 실행 확인: python gripper_server.py --port 5009"
        )

    # 2) homing (범위 캘리브레이션)
    print("[gripper-init] homing 중... (30초 이내)")
    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-init] homing 실패")

    # [디버그] homing 결과로 얻은 min_q / max_q 확인
    _min_q = getattr(gripper_obj, "min_q", None)
    _max_q = getattr(gripper_obj, "max_q", None)
    if _min_q is None or _max_q is None:
        raise RuntimeError(
            "[gripper-init] homing 성공했지만 min_q/max_q가 없음.\n"
            "  gripper_server.py 응답 형식에 min_q/max_q 필드가 있는지 확인하세요."
        )
    print(f"[gripper-init] homing 완료 | min_q={_min_q}  max_q={_max_q}")

    # 3) 제어 루프 시작
    print("[gripper-init] start() 호출 중...")
    gripper_obj.start()
    print("[gripper-init] start() 완료")

    # 4) 완전 열기 (normalized_q=1.0 → OPEN) — wait_for_reply=True로 확인
    print("[gripper-init] 완전 열기 명령 전송 (normalized=1.0)...")
    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(1.0)  # 그리퍼 물리적 이동 완료 대기 (0.3s → 1.0s)

    # [디버그] 현재 그리퍼 상태 확인
    try:
        _grip_state = gripper_obj.get_state()
        print(f"[gripper-init] 현재 그리퍼 상태: {_grip_state}  (열기 후 확인)")
    except Exception as _se:
        print(f"[gripper-init][WARN] 상태 조회 실패: {_se}")

    # [디버그] 현재 normalized target 확인
    try:
        _grip_norm = gripper_obj.get_normalized_target()
        print(f"[gripper-init] 현재 normalized target: {_grip_norm}  (1.0=OPEN 이어야 함)")
    except Exception as _ne:
        print(f"[gripper-init][WARN] normalized target 조회 실패: {_ne}")

    GRIPPER_INIT_DONE = True
    print("[gripper-init] ✅ 완료 — env 생성 시 이 객체를 재사용합니다.")


---
## 📊 [시각화] Action 로그 플롯 — 관절 궤적 및 Chunk 경계 분석

Episode loop 실행 후 `_log_*` 버퍼를 기반으로 3종류 그래프를 생성합니다.

| 그래프 | 내용 |
|--------|------|
| ① 전체 궤적 | 관절별 Commanded vs Actual (deg) |
| ② Step Δ norm | Chunk 경계에서 급변(spike) 여부 확인 |
| ③ Chunk 경계 zoom-in | 경계 전후 ±N step 궤적 확대 비교 |

> Episode loop (`▶️ Step 5`) 실행 후에만 동작합니다.

In [ ]:
# ---------- (cell 10) Action chunk 로그 시각화 — chunk 경계 튀는 현상 분석 ----------
# Cell 9 (episode loop) 실행 후 _log_* 버퍼를 기반으로 3종류 plot 생성:
#   ① 전체 에피소드: 관절별 commanded vs actual 궤적
#   ② step-to-step Δ norm: chunk 경계에서 spike 여부 확인
#   ③ chunk 경계 zoom-in: 경계 전후 궤적 확대 비교
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if "_log_cmd_right" not in globals() or len(_log_cmd_right) == 0:
    print("[plot] 로그 데이터 없음 — Cell 9 (episode loop)를 먼저 실행하세요.")
else:
    # --------------------------------------------------
    # 데이터 준비
    # --------------------------------------------------
    cmd_r    = np.array(_log_cmd_right)     # (N, 7)  rad
    cmd_l    = np.array(_log_cmd_left)      # (N, 7)  rad
    grip_r   = np.array(_log_cmd_grip_r)    # (N,)
    grip_l   = np.array(_log_cmd_grip_l)    # (N,)
    act_r    = np.array(_log_actual_right) if _log_actual_right else np.empty((0, 7))
    act_l    = np.array(_log_actual_left)  if _log_actual_left  else np.empty((0, 7))
    act_x    = np.array(_log_actual_step,  dtype=int)   # (M,)
    c_starts = np.array(_log_chunk_start,  dtype=int)   # (C,)

    N = cmd_r.shape[0]
    steps_x = np.arange(N)

    print(f"[plot] 총 step: {N}  |  청크 수: {len(c_starts)}  |  진단 샘플: {len(act_x)}")
    print(f"[plot] 청크 시작 step: {c_starts.tolist()}")

    # --------------------------------------------------
    # ① 전체 궤적 — 관절별 commanded vs actual (deg)
    # --------------------------------------------------
    def _plot_arm_full(title, cmd, act, act_steps, grip, tag):
        fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
        fig.suptitle(title, fontsize=13, fontweight="bold")
        axes_flat = axes.flatten()

        for j in range(7):
            ax = axes_flat[j]
            ax.plot(steps_x, np.degrees(cmd[:, j]),
                    color="steelblue", lw=1.2, label="commanded")
            if len(act_steps) > 0:
                ax.scatter(act_steps, np.degrees(act[:, j]),
                           color="tomato", s=18, zorder=5, label="actual")
            for cs in c_starts:
                ax.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
            ax.set_title(f"{tag}_J{j}", fontsize=10)
            ax.set_ylabel("deg", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(True, alpha=0.3)

        ax_g = axes_flat[7]
        ax_g.plot(steps_x, grip, color="darkorange", lw=1.2)
        for cs in c_starts:
            ax_g.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
        ax_g.set_title(f"{tag}_Gripper", fontsize=10)
        ax_g.set_ylabel("value", fontsize=8)
        ax_g.tick_params(labelsize=7)
        ax_g.grid(True, alpha=0.3)

        for ax in axes[1]:
            ax.set_xlabel("step", fontsize=8)

        handles = [
            mpatches.Patch(color="steelblue", label="commanded"),
            mpatches.Patch(color="tomato",    label="actual (sampled)"),
            plt.Line2D([0], [0], color="dimgray", linestyle="--", label="chunk start"),
        ]
        fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9,
                   bbox_to_anchor=(0.5, -0.01))
        plt.tight_layout(rect=[0, 0.04, 1, 1])
        plt.show()

    _plot_arm_full("Right Arm — Commanded vs Actual (deg)", cmd_r, act_r, act_x, grip_r, "R")
    _plot_arm_full("Left Arm  — Commanded vs Actual (deg)", cmd_l, act_l, act_x, grip_l, "L")

    # --------------------------------------------------
    # ② Step-to-step Δ norm — chunk 경계 spike 확인
    # --------------------------------------------------
    delta_r = np.linalg.norm(np.diff(cmd_r, axis=0), axis=1)   # (N-1,)
    delta_l = np.linalg.norm(np.diff(cmd_l, axis=0), axis=1)

    fig2, (ax2r, ax2l) = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
    fig2.suptitle(
        "Step-to-step Command Δ (L2 norm, rad) — chunk 경계 spike 확인\n"
        "빨간 점선: chunk 마지막 step → 다음 chunk 첫 step (경계 전환 위치)",
        fontsize=11, fontweight="bold"
    )
    for ax2, delta, color, arm_tag in [
        (ax2r, delta_r, "steelblue", "Right"),
        (ax2l, delta_l, "coral",     "Left"),
    ]:
        ax2.plot(np.arange(N - 1), delta, color=color, lw=0.9, label=f"{arm_tag} Δ")
        for k, cs in enumerate(c_starts[1:]):
            ax2.axvline(x=cs - 1, color="red", linestyle="--", lw=1.5, alpha=0.85,
                        label="chunk boundary" if k == 0 else None)
        ax2.set_ylabel("Δ norm (rad)", fontsize=9)
        ax2.set_title(f"{arm_tag} Arm", fontsize=10)
        ax2.grid(True, alpha=0.3)
        ax2.legend(fontsize=8)
    ax2l.set_xlabel("step", fontsize=9)
    plt.tight_layout()
    plt.show()

    # 경계 delta 통계
    print("\n[plot] === Step Δ 통계 (chunk 경계 vs 내부) ===")
    for arm_tag, delta in [("Right", delta_r), ("Left", delta_l)]:
        in_mask = np.ones(len(delta), dtype=bool)
        for cs in c_starts[1:]:
            if 0 < cs - 1 < len(in_mask):
                in_mask[cs - 1] = False   # chunk 경계 직전 step
        b_delta = delta[~in_mask]
        i_delta = delta[in_mask]
        print(
            f"  {arm_tag}: in-chunk avg={i_delta.mean():.4f} rad | "
            f"boundary avg={b_delta.mean() if len(b_delta) else 0:.4f} rad | "
            f"boundary max={b_delta.max()  if len(b_delta) else 0:.4f} rad"
        )

    # --------------------------------------------------
    # ③ Chunk 경계 zoom-in (최대 4개 경계)
    # --------------------------------------------------
    if len(c_starts) > 1:
        ZOOM = 15   # 경계 앞뒤 step 수
        n_boundary = len(c_starts) - 1
        n_show = min(n_boundary, 4)

        fig3, axes3 = plt.subplots(n_show, 2, figsize=(16, 4 * n_show))
        if n_show == 1:
            axes3 = axes3[np.newaxis, :]
        fig3.suptitle(
            f"Chunk Boundary Zoom-in (경계 ±{ZOOM} step) — 궤적 급변 여부 확인",
            fontsize=12, fontweight="bold"
        )

        for bi in range(n_show):
            bstep = int(c_starts[bi + 1])
            lo = max(0, bstep - ZOOM)
            hi = min(N, bstep + ZOOM)
            xs_z = np.arange(lo, hi)

            for col, (cmd_arm, act_arm, arm_tag) in enumerate([
                (cmd_r, act_r, "Right"),
                (cmd_l, act_l, "Left"),
            ]):
                ax = axes3[bi, col]
                for j in range(7):
                    ax.plot(xs_z, np.degrees(cmd_arm[lo:hi, j]),
                            lw=1.3, alpha=0.85, label=f"J{j}")
                # 실제 위치 샘플 (해당 구간)
                if len(act_x) > 0:
                    mask = (act_x >= lo) & (act_x < hi)
                    if mask.any():
                        for j in range(7):
                            ax.scatter(act_x[mask], np.degrees(act_arm[mask, j]),
                                       s=30, marker="x", zorder=6)
                ax.axvline(x=bstep, color="red", lw=2.2, linestyle="-",
                           label=f"boundary (chunk {bi+1}→{bi+2})")
                # chunk 내 anchor 위치 표시 (경계 직후 첫 스텝)
                if lo <= bstep < hi:
                    ax.axvspan(bstep - 0.5, bstep + 0.5, color="red", alpha=0.12)
                ax.set_title(
                    f"{arm_tag} | boundary @ step {bstep}  (chunk {bi+1} → {bi+2})",
                    fontsize=10
                )
                ax.set_xlabel("step", fontsize=8)
                ax.set_ylabel("deg", fontsize=8)
                ax.tick_params(labelsize=7)
                ax.grid(True, alpha=0.3)
                ax.legend(fontsize=7, loc="upper right", ncol=4)

        plt.tight_layout()
        plt.show()

    # --------------------------------------------------
    # 분석 가이드
    # --------------------------------------------------
    print("\n[plot] ===== 튀는 현상 원인 분석 가이드 =====")
    print("  ② Δ 그래프에서 chunk 경계(빨간 점선)만 spike → anchor_relative 재계산 시 방향 불일치")
    print("     └ 이전 chunk 마지막 target과 새 chunk anchor 위치 간 gap 발생")
    print("  ② 경계와 무관하게 전반적으로 Δ가 큼   → EXECUTE_CHUNK_SIZE 또는 REPLAY_DT가 너무 짧음")
    print("  ③ zoom-in에서 경계 직후 궤적이 급변   → 새 inference의 예측 방향이 이전과 다름")
    print("  ③ 경계 직전/직후 commanded가 연속적  → 로봇이 command를 따라가지 못하는 tracking 문제")
    print("  대응:")
    print("    - spike 심함: EXECUTE_CHUNK_SIZE 늘리거나 REPLAY_DT 줄이기")
    print("    - tracking_err 큼: REPLAY_DT 늘리거나 minimum_time 증가")
    print("    - 방향 불일치: anchor_relative 확인 / 추론 설정 확인")


## 🖼️ [시각화] Policy 입력 RGB 이미지 프레임 확인

Episode loop 실행 중 매 inference 시점에 캡처된 카메라 프레임을 시각화합니다.

- **행**: 카메라 종류 (Head / Left Wrist / Right Wrist)
- **열**: inference 시간 순서
- `FRAME_STEP` : 표시 간격 (1=모든 프레임, 5=5번에 1번)
- `MAX_FRAMES` : 최대 열 수 제한
- `SAVE_DIR` : 경로 지정 시 PNG로 저장

> Episode loop (`▶️ Step 5`) 실행 후에만 동작합니다.

In [ ]:
# ---------- Policy 입력 RGB 이미지 프레임 시각화 ----------
# episode loop 실행 후 _log_obs_images 버퍼를 기반으로 각 inference 시점의 카메라 프레임을 시각화.
#
# 설정:
#   FRAME_STEP     : 표시할 inference 프레임 간격 (1=모든 inference, 5=5번에 1번)
#   MAX_FRAMES     : 최대 표시 열 수 (None이면 전체)
#   SAVE_DIR       : 이미지 저장 디렉터리 (None이면 저장 안 함)
import matplotlib.pyplot as plt
import os

FRAME_STEP  = 1     # inference 프레임 샘플링 간격
MAX_FRAMES  = 20    # 표시할 최대 열(timestep) 수 — None이면 전체
SAVE_DIR    = None  # e.g. "/tmp/obs_frames" — None이면 저장 안 함

_OBS_KEY_LABELS = {
    "observation/head_image":        "Head",
    "observation/left_wrist_image":  "Left Wrist",
    "observation/right_wrist_image": "Right Wrist",
}

if "_log_obs_images" not in globals() or len(_log_obs_images) == 0:
    print("[img-plot] 이미지 로그 없음 — episode loop를 먼저 실행하세요.")
else:
    # --------------------------------------------------
    # 표시할 프레임 선택
    # --------------------------------------------------
    sampled = _log_obs_images[::FRAME_STEP]
    if MAX_FRAMES is not None:
        sampled = sampled[:MAX_FRAMES]

    # 실제로 이미지가 들어있는 key만 사용
    present_keys = [k for k in _OBS_KEY_LABELS if k in sampled[0]]
    n_rows = len(present_keys)
    n_cols = len(sampled)

    print(f"[img-plot] inference 총 {len(_log_obs_images)}회  →  표시: {n_cols} timestep × {n_rows} 카메라")

    if n_rows == 0:
        print("[img-plot] 관측 딕셔너리에 이미지 키가 없습니다.")
    else:
        FIG_W = max(14, n_cols * 2.0)
        FIG_H = n_rows * 2.5
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(FIG_W, FIG_H),
                                 squeeze=False)
        fig.suptitle(
            f"Policy 입력 RGB 프레임  (inference step 간격: {FRAME_STEP}, "
            f"표시: {n_cols}/{len(_log_obs_images)} steps)",
            fontsize=12, fontweight="bold"
        )

        for row_idx, cam_key in enumerate(present_keys):
            cam_label = _OBS_KEY_LABELS[cam_key]
            for col_idx, snap in enumerate(sampled):
                ax = axes[row_idx][col_idx]
                img = snap.get(cam_key)
                if img is not None:
                    ax.imshow(img)
                else:
                    ax.set_facecolor("#222")
                    ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                            transform=ax.transAxes, color="white", fontsize=8)
                ax.axis("off")

                # 상단 행에만 step 번호 표시
                if row_idx == 0:
                    ax.set_title(f"t={snap['step']}", fontsize=7, pad=2)

            # 좌측 행 레이블
            axes[row_idx][0].set_ylabel(cam_label, fontsize=9, rotation=90,
                                        labelpad=4, va="center")

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()

        # --------------------------------------------------
        # 선택적 저장
        # --------------------------------------------------
        if SAVE_DIR is not None:
            os.makedirs(SAVE_DIR, exist_ok=True)
            for snap in _log_obs_images:
                step = snap["step"]
                for cam_key in present_keys:
                    img = snap.get(cam_key)
                    if img is None:
                        continue
                    cam_tag = cam_key.split("/")[-1]   # e.g. "head_image"
                    fname = os.path.join(SAVE_DIR, f"step{step:04d}_{cam_tag}.png")
                    plt.imsave(fname, img)
            print(f"[img-plot] {len(_log_obs_images) * len(present_keys)}장 저장 완료 → {SAVE_DIR}")

    # --------------------------------------------------
    # 단일 timestep 확대 보기 (마지막 inference 프레임)
    # --------------------------------------------------
    last_snap = _log_obs_images[-1]
    present_last = [k for k in _OBS_KEY_LABELS if k in last_snap]
    if present_last:
        fig2, axes2 = plt.subplots(1, len(present_last), figsize=(5 * len(present_last), 4))
        if len(present_last) == 1:
            axes2 = [axes2]
        fig2.suptitle(
            f"마지막 inference 프레임  (step={last_snap['step']})",
            fontsize=11, fontweight="bold"
        )
        for ax2, cam_key in zip(axes2, present_last):
            img2 = last_snap.get(cam_key)
            if img2 is not None:
                ax2.imshow(img2)
            ax2.set_title(_OBS_KEY_LABELS[cam_key], fontsize=10)
            ax2.axis("off")
        plt.tight_layout()
        plt.show()


---
## 🧹 Step 6 — 환경 및 Runtime 정리

에피소드 종료 후 `Runtime`과 `RBY1Environment`를 안전하게 닫고 상태 플래그를 초기화합니다.

- `runtime.mark_episode_complete()` → episode 완료 신호 전송
- `env.close()` → 카메라 파이프라인 및 로봇 연결 해제

> **종료 시 반드시 실행하세요.** 실행하지 않으면 다음 실행 시 RealSense "already in use" 오류가 발생할 수 있습니다.

In [ ]:
# ---------- Step 6: 정리 ----------
if "runtime" not in globals():
    print("[stop] runtime 없음")
else:
    try:
        runtime.mark_episode_complete()
    except Exception as exc:
        print(f"[stop][WARN] mark_episode_complete 실패: {exc}")

    if "runtime_thread" in globals() and runtime_thread is not None:
        try:
            runtime_thread.join(timeout=5.0)
        except Exception as exc:
            print(f"[stop][WARN] thread join 실패: {exc}")

if "env" in globals() and env is not None:
    try:
        env.close()
        print("[stop] environment closed")
    except Exception as exc:
        print(f"[stop][WARN] env.close 실패: {exc}")

ENV_SETUP_DONE = False
RUNTIME_RUNNING = False
runtime_thread = None
print("[stop] done")